<a href="https://colab.research.google.com/github/IshiPareek/mech_interpet/blob/main/03_gpt2_activation_patching_on_io.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — imports
!pip install transformer_lens
import transformer_lens
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
import torch
import numpy as np
from jaxtyping import Float, Int
import einops

print(f"TransformerLens installd")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
model = HookedTransformer.from_pretrained("gpt2")

In [ ]:
tokens = model.to_tokens("The meaning of life is")
str_tokens = model.to_str_tokens(tokens)
print(model.to_str_tokens(tokens))
logits = model(tokens)
print(logits.shape)

In [ ]:
# ── VISUALIZATION BOILERPLATE ── define once, call anywhere ──────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def viz(cache, plot_type, layer=0, head=0, dims=64, model=model, str_tokens=str_tokens, original_loss=None, ablated_loss=None):
    """
    Universal cache visualizer.

    plot_type options:
        "embed"      → token embeddings
        "pos_embed"  → positional embeddings
        "pattern"    → attention pattern for one head
        "all_heads"  → all 12 heads in a layer
        "resid"      → residual stream at a layer
        "logit_lens" → top predicted token across all layers
        "qkv"        → query, key, value vectors
    """

    def heatmap(ax, matrix, title, row_labels=None, cmap="RdBu_r"):
        vmax = np.abs(matrix).max()
        ax.imshow(matrix, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_title(title, fontsize=10, pad=6)
        if row_labels:
            ax.set_yticks(range(len(row_labels)))
            ax.set_yticklabels(row_labels, fontsize=8)
        else:
            ax.set_yticks([])
        ax.set_xticks([])

    # ── embed ────────────────────────────────────────────────────────────────
    if plot_type == "embed":
        data = cache["hook_embed"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_embed — first {dims} dims", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

    # ── pos_embed ────────────────────────────────────────────────────────────
    elif plot_type == "pos_embed":
        data = cache["hook_pos_embed"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_pos_embed — first {dims} dims", row_labels=str_tokens, cmap="PiYG")
        plt.tight_layout(); plt.show()

    # ── pattern ──────────────────────────────────────────────────────────────
    elif plot_type == "pattern":
        data = cache[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().numpy()
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(str_tokens))); ax.set_xticklabels(str_tokens, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(str_tokens))); ax.set_yticklabels(str_tokens, fontsize=8)
        ax.set_title(f"attention pattern — layer {layer}, head {head}", fontsize=10)
        plt.tight_layout(); plt.show()

    # ── all_heads ────────────────────────────────────────────────────────────
    elif plot_type == "all_heads":
        fig, axes = plt.subplots(3, 4, figsize=(14, 9))
        for h, ax in enumerate(axes.flat):
            data = cache[f"blocks.{layer}.attn.hook_pattern"][0, h].detach().numpy()
            ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
            ax.set_title(f"H{h}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle(f"all attention heads — layer {layer}", fontsize=12)
        plt.tight_layout(); plt.show()

    # ── resid ────────────────────────────────────────────────────────────────
    elif plot_type == "resid":
        data = cache[f"blocks.{layer}.hook_resid_post"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_resid_post — layer {layer}, first {dims} dims", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

    # ── logit_lens ───────────────────────────────────────────────────────────
    elif plot_type == "logit_lens":
        n_layers = model.cfg.n_layers
        cell_text = []
        for l in range(n_layers):
            resid = cache[f"blocks.{l}.hook_resid_post"]
            top = model.unembed(model.ln_final(resid))[0, :, :].argmax(dim=-1)
            cell_text.append([model.to_string(t) for t in top])
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.axis("off")
        tbl = ax.table(cellText=cell_text, rowLabels=[f"L{l}" for l in range(n_layers)],
                       colLabels=str_tokens, loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.3)
        ax.set_title("logit lens — top predicted token per layer × position", fontsize=11, pad=20)
        plt.tight_layout(); plt.show()

    # ── qkv ──────────────────────────────────────────────────────────────────
    elif plot_type == "qkv":
        fig, axes = plt.subplots(1, 3, figsize=(14, 3))
        for ax, key, label in zip(axes, ["hook_q","hook_k","hook_v"], ["Query","Key","Value"]):
            data = cache[f"blocks.{layer}.attn.{key}"][0, :, head, :].detach().numpy()
            heatmap(ax, data, f"{label} — layer {layer}, head {head}", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

## ABLATION
    elif plot_type == "ablation":
      if original_loss is None or ablated_loss is None :
        print ("pass original_loss and ablation_loss to use this plot type")
        return

      orig = original_loss.item()
      abla = ablated_loss.item()
      diff = abla - orig

      fig, ax = plt.subplots(figsize=(5, 4))
      bars = ax.bar(
      ["Original Loss", "Ablated Loss"],
      [orig, abla],
      color=["steelblue", "tomato"])

      ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
      ax.set_title(
            f"Head Ablation — Layer {layer}, Head {head}\nΔ Loss = +{diff:.3f}",
      fontsize=11)
      ax.set_ylabel("Cross Entropy Loss")
      ax.set_ylim(0, max(orig, abla) * 1.2)
      plt.tight_layout(); plt.show()

    else:
        print(f"unknown plot_type '{plot_type}'. options: embed, pos_embed, pattern, all_heads, resid, logit_lens, qkv")

In [ ]:
logits, cache = model.run_with_cache(tokens, remove_batch_dim=True)
print(list(cache.keys())[:10])

In [ ]:
from functools import partial
import tqdm.auto as tqdm

# ── prompts ──────────────────────────────────────────────────────────────
clean_prompt = "After John and Mary went to the store, Mary gave a bottle of milk to"
corrupted_prompt = "After John and Mary went to the store, John gave a bottle of milk to"

clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

# ── logit diff metric ─────────────────────────────────────────────────────
def logits_to_logit_diff(logits, correct_answer=" John", incorrect_answer=" Mary"):
    correct_index = model.to_single_token(correct_answer)
    incorrect_index = model.to_single_token(incorrect_answer)
    return logits[0, -1, correct_index] - logits[0, -1, incorrect_index]

# ── run clean and corrupted ───────────────────────────────────────────────
clean_logits, clean_cache = model.run_with_cache(clean_tokens)
clean_logit_diff = logits_to_logit_diff(clean_logits)
print(f"Clean logit difference:     {clean_logit_diff.item():.3f}")

corrupted_logits = model(corrupted_tokens)
corrupted_logit_diff = logits_to_logit_diff(corrupted_logits)
print(f"Corrupted logit difference: {corrupted_logit_diff.item():.3f}")

# ── patching hook ─────────────────────────────────────────────────────────
def residual_stream_patching_hook(
    resid_pre: Float[torch.Tensor, "batch pos d_model"], hook: HookPoint, position: int
) -> Float[torch.Tensor, "batch pos d_model"]:
    clean_resid_pre = clean_cache[hook.name]
    resid_pre[:, position, :] = clean_resid_pre[:, position, :]
    return resid_pre

# ── patching loop ─────────────────────────────────────────────────────────
num_positions = len(clean_tokens[0])
ioi_patching_result = torch.zeros((model.cfg.n_layers, num_positions), device=model.cfg.device)

for layer in tqdm.tqdm(range(model.cfg.n_layers)):
    for position in range(num_positions):
        temp_hook_fn = partial(residual_stream_patching_hook, position=position)
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(utils.get_act_name("resid_pre", layer), temp_hook_fn)]
        )
        patched_logit_diff = logits_to_logit_diff(patched_logits).detach()
        ioi_patching_result[layer, position] = (patched_logit_diff - corrupted_logit_diff) / (
            clean_logit_diff - corrupted_logit_diff
        )

# ── plot ──────────────────────────────────────────────────────────────────
str_tokens = model.to_str_tokens(clean_tokens)
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(ioi_patching_result.detach().cpu().numpy(), cmap="RdBu_r", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, label="Logit Diff Recovery (0=corrupted, 1=clean)")
ax.set_xlabel("Token Position")
ax.set_ylabel("Layer")
ax.set_title("Activation Patching — Residual Stream\nwhich layer+position matters for IOI?")
ax.set_xticks(range(num_positions))
ax.set_xticklabels(str_tokens, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(model.cfg.n_layers))
ax.set_yticklabels([f"L{l}" for l in range(model.cfg.n_layers)])
plt.tight_layout()
plt.show()